In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [54]:
df = pd.read_json("../../../data/post_quality/raw/post_data_v4.json")
df.reset_index(drop=True)
df.head()

,id,title,body,effort,openness,is_confident,subreddit,tag
0,1,If your first-ever attempt at gambling went co...,,0,0,True,r/Showerthoughts,NaN
1,2,"According to birds, all land animals are botto...",,0,0,True,r/Showerthoughts,NaN
2,3,Very few people can actually keep their watch ...,,0,0,True,r/Showerthoughts,NaN
3,4,"What is keeping the really deadly diseases, li...",,0,0,True,r/askscience,NaN
4,5,When did humans start living indoors?,,0,0,True,r/askscience,NaN


In [55]:
df.shape

(240, 8)

# Checking for null values and dropping the unrelevent column

In [56]:
for col in df.columns:
    print(f"Number of null value in '{col}' column is {df[col].isnull().sum()}")

Number of null value in 'id' column is 0
Number of null value in 'title' column is 0
Number of null value in 'body' column is 0
Number of null value in 'effort' column is 0
Number of null value in 'openness' column is 0
Number of null value in 'is_confident' column is 0
Number of null value in 'subreddit' column is 0
Number of null value in 'tag' column is 239


In [57]:
df.drop(columns=['tag','effort',"id","is_confident"],inplace=True)

In [58]:
df.shape

(240, 4)

In [59]:
for col in df.columns:
    print(f"Number of null value in '{col}' column is {df[col].isnull().sum()}")

Number of null value in 'title' column is 0
Number of null value in 'body' column is 0
Number of null value in 'openness' column is 0
Number of null value in 'subreddit' column is 0


# Try Getting baseline feature set like this to just check whether any signals are there or not. I will include features are:
1. Explicit Question Presence(`has_question_mark`):
    1. Why
        - Questions are the most direct openness signal.
        - But we already know: question is not openness.
        - This establishes a weak baseline.
    2. Hypothesis
        Posts without questions are less open on average.

2. Question Count(`num_questions`):
    1. Why
        - Multiple questions often indicate exploration
        - But too many can also be ranting
    2. Hypothesis
        - Higher question density correlates with openness, but with diminishing returns.
3. Uncertainity/Hedging Language(`has_uncertainty_marker`, `uncertainty_count`):
    1. Why
        - This is the core openness signal
        - Expressed uncertainty = epistemic openness
    2. Hypothesis
        Explicit uncertainty strongly correlates with openness.
4. Invitation/Solicitation Phrases(`has_invitation_phrase`):
    1. Why
        - Distinguishes "asking" from "asserting"
        - Especially useful against rhetorical questions
    2. Hypothesis
        - Direct invitation are high-precision openness indicators
5. Assertion Density(Anti-Openness)(`assertion_marker_count`):
    1. Why
        - Openness is as much about absence of certainty
        - High assertion density -> low openness.
    2. Hypothesis
        - Strong assertions reduce openness likelihood.
6. First-person Reflection Markers(`has_reflective_phrase`):
    1. Why
        - Reflection != effort
        - Reflection is somewhat correlates with openness
    2. Hypothesis
        - Self-reflective language correlates with openness.


In [60]:
df["text"] = (df["title"].fillna("") + " " + df["body"].fillna("")).str.lower()


In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   title      240 non-null    object
 1   body       240 non-null    object
 2   openness   240 non-null    int64 
 3   subreddit  240 non-null    object
 4   text       240 non-null    object
dtypes: int64(1), object(4)
memory usage: 9.5+ KB


In [62]:
import re

def has_question_mark(text):
    if not isinstance(text,str):
        return 0
    return int("?" in text)

df["has_question_mark"] = df['text'].apply(has_question_mark)


def num_questions(text):
    if not isinstance(text,str):
        return 0
    return text.count("?")

df['num_questions'] = df['text'].apply(num_questions)


UNCERTAINTY_MARKERS = [
    "maybe", "might", "not sure", "i think", "i feel",
    "i guess", "i'm unsure", "i am unsure",
    "could be wrong", "not certain", "possibly"
]

def uncertainty_count(text):
    if not isinstance(text,str):
        return 0
    return sum(1 for m in UNCERTAINTY_MARKERS if m in text)

def has_uncertainty_marker(text):
    return int(uncertainty_count(text) > 0)

df["uncertainty_count"] = df["text"].apply(uncertainty_count)
df["has_uncertainty_marker"] = df["text"].apply(has_uncertainty_marker)


INVITATION_PHRASES = [
    "what do you think",
    "any advice",
    "any thoughts",
    "can someone explain",
    "help me understand",
    "thoughts?",
    "does anyone know",
    "am i missing something"
]

def has_invitation_phrase(text):
    if not isinstance(text,str):
        return 0
    return int(any(p in text for p in INVITATION_PHRASES))

df['has_invitation_phrase'] = df['text'].apply(has_invitation_phrase)


ASSERTION_MARKERS = [
    "obviously",
    "everyone knows",
    "the truth is",
    "there is no doubt",
    "clearly",
    "this proves",
    "it's obvious",
    "without question"
]

def assertion_marker_count(text):
    if not isinstance(text,str):
        return 0
    return sum(1 for m in ASSERTION_MARKERS if m in text)

df["assertion_marker_count"] = df["text"].apply(assertion_marker_count)


REFLECTIVE_PHRASES = [
    "i'm trying to understand",
    "i am trying to understand",
    "i want to understand",
    "i'm learning",
    "i am learning",
    "i'm confused",
    "i am confused",
    "i don't fully understand",
    "i do not fully understand"
]

def has_reflective_phrase(text):
    if not isinstance(text,str):
        return 0
    return int(any(p in text for p in REFLECTIVE_PHRASES))

df["has_reflective_phrase"] = df["text"].apply(has_reflective_phrase)

In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   title                   240 non-null    object
 1   body                    240 non-null    object
 2   openness                240 non-null    int64 
 3   subreddit               240 non-null    object
 4   text                    240 non-null    object
 5   has_question_mark       240 non-null    int64 
 6   num_questions           240 non-null    int64 
 7   uncertainty_count       240 non-null    int64 
 8   has_uncertainty_marker  240 non-null    int64 
 9   has_invitation_phrase   240 non-null    int64 
 10  assertion_marker_count  240 non-null    int64 
 11  has_reflective_phrase   240 non-null    int64 
dtypes: int64(8), object(4)
memory usage: 22.6+ KB


In [64]:
df.tail()

,title,body,openness,subreddit,text,has_question_mark,num_questions,uncertainty_count,has_uncertainty_marker,has_invitation_phrase,assertion_marker_count,has_reflective_phrase
235,Does being an introvert actually lead to highe...,There’s a common belief that introverts tend t...,1,r/TrueAskReddit,does being an introvert actually lead to highe...,1,3,0,0,0,0,0
236,Does anything seem legendary anymore?,I was having a conversation with a friend as t...,1,r/TrueAskReddit,does anything seem legendary anymore? i was ha...,1,3,2,1,0,0,0
237,How will US government deal with a large propo...,How will US government deal with a large propo...,1,r/TrueAskReddit,how will us government deal with a large propo...,1,6,0,0,0,0,0
238,Why so many straight men are constantly polici...,I’ve noticed a pattern where straight guys lov...,1,r/TrueAskReddit,why so many straight men are constantly polici...,1,4,0,0,0,0,0
239,Are we done?,Imagine the year is 2050 AI has evolved into A...,1,r/TrueAskReddit,are we done? imagine the year is 2050 ai has e...,1,4,0,0,0,0,0


In [65]:
df["has_reflective_phrase"].value_counts()

has_reflective_phrase
0    238
1      2
Name: count, dtype: int64

In [66]:
df["assertion_marker_count"].value_counts()

assertion_marker_count
0    224
1     16
Name: count, dtype: int64

In [67]:
df["has_invitation_phrase"].value_counts()

has_invitation_phrase
0    227
1     13
Name: count, dtype: int64

In [68]:
df["has_uncertainty_marker"].value_counts()

has_uncertainty_marker
0    183
1     57
Name: count, dtype: int64

# Observation: 
* For these features `has_invitation_phrase`, `assertion_marker_count`, `has_reflective_phrase`, `has_uncertainty_marker`, all are predominantly 0s in the dataset

## Freezing feature list and start running ML model

In [69]:
## Let's build a helper function to make error_df
import pandas as pd

def build_error_df(df_test, y_true, y_pred, model_name):
    out = df_test.copy()
    out['true_label'] = y_true
    out['pred_label'] = y_pred
    out['correct'] = y_true == y_pred
    out['model'] = model_name
    return out

In [70]:
## Generic function to split df in same test and train
from sklearn.model_selection import train_test_split


def split_df(
    df,
    label_col,
    test_size=0.2,
    random_state=52,
    stratify=True
):
    """
    Splits dataframe into train and test sets.

    Returns
    -------
    df_train : pd.DataFrame
    df_test : pd.DataFrame
    """

    df = df.reset_index(drop=True)

    stratify_col = df[label_col] if stratify else None

    train_idx, test_idx = train_test_split(
        df.index,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_col
    )

    df_train = df.loc[train_idx]
    df_test = df.loc[test_idx]

    return df_train, df_test


In [71]:
## Running numerical only model
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score


def run_numerical_model(
    df_train,
    df_test,
    label_col,
    num_cols=None,
    model=None,
    scaler=None,
    model_name="numerical_only"
):
    """
    Trains and evaluates a numerical-only classification model.

    Returns
    -------
    dict with:
        - model
        - scaler
        - classification_report
        - errors_df
        - y_pred
    """

    if num_cols is None:
        num_cols = df_train.select_dtypes(include="number").columns.tolist()
        num_cols.remove(label_col)

    if scaler is None:
        scaler = StandardScaler()

    if model is None:
        model = LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )

    X_train = scaler.fit_transform(df_train[num_cols])
    X_test = scaler.transform(df_test[num_cols])

    y_train = df_train[label_col]
    y_test = df_test[label_col]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    report = classification_report(y_test, y_pred)

    errors_df = build_error_df(
        df_test=df_test,
        y_true=y_test,
        y_pred=y_pred,
        model_name=model_name
    )

    errors_df['error_type'] = ''
    errors_df['error_subtype'] = ''
    errors_df['notes'] = ''

    return {
        "model": model,
        "scaler": scaler,
        "classification_report": report,
        "errors_df": errors_df,
        "accuracy":accuracy_score(y_test,y_pred),
        "y_pred": y_pred
    }


In [72]:
## Function to get error across k-folds
import pandas as pd
from sklearn.model_selection import StratifiedKFold

def run_k_fold_error_analysis(df:pd.DataFrame, n_splits=5, label_col='label'):
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )
    
    all_errors = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(df, df[label_col])):
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()
        
        result = run_numerical_model(
            train_df,
            test_df,
            label_col=label_col
        )
        
        errors_df = result['errors_df'].copy()
        
        errors_df['fold'] = fold
        
        all_errors.append(errors_df)
    
    errors_all = pd.concat(all_errors).reset_index(drop=True)
    
    errors_only = errors_all[errors_all['correct'] == False]
    
    errors_all['error_type'] = 'N/A'  
    errors_all['error_subtype'] = 'N/A'  
    errors_all['notes'] = ''  
    
    true_label_counts = errors_only['true_label'].value_counts()
    pred_label_counts = errors_only['pred_label'].value_counts()
    error_group_counts = errors_only.groupby(['true_label', 'pred_label']).size()
    
    return {
        'errors_all': errors_all,
        'errors_only': errors_only,
        'true_label_counts': true_label_counts,
        'pred_label_counts': pred_label_counts,
        'error_group_counts': error_group_counts
    }


In [73]:
result_open_v1 = run_k_fold_error_analysis(df=df,label_col="openness")

In [74]:
errors_v1 = result_open_v1['errors_only']
errors_v1

,title,body,openness,subreddit,text,has_question_mark,num_questions,uncertainty_count,has_uncertainty_marker,has_invitation_phrase,assertion_marker_count,has_reflective_phrase,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold
12,Java or Javascript or C#?,"I'm not sure which one if these is better, to ...",0,r/learningprogramming,java or javascript or c#? i'm not sure which o...,1,2,1,1,0,0,0,0,1,False,numerical_only,,,,0
14,Why are under cabinet strip lights so complica...,It seems crazy there is no consensus on how to...,0,r/HomeImprovement,why are under cabinet strip lights so complica...,1,5,0,0,0,0,0,0,1,False,numerical_only,,,,0
15,Is insulation between floors worth it during c...,my builder is charging $2500 for R-15 3.5 inch...,0,r/HomeImprovement,is insulation between floors worth it during c...,1,2,0,0,0,0,0,0,1,False,numerical_only,,,,0
17,Which companies now only offer on site softwar...,Hi all- currently work remotely and I now real...,0,r/cscareerquestions,which companies now only offer on site softwar...,1,2,0,0,0,0,0,0,1,False,numerical_only,,,,0
18,Do all flowering plants share a common ancesto...,With the variety of flowering plants in the wo...,0,r/askscience,do all flowering plants share a common ancesto...,1,5,0,0,0,0,0,0,1,False,numerical_only,,,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,How exactly is gas stored in soda before you o...,Hey scientists!\nMaybe this is a super basic q...,0,r/askscience,how exactly is gas stored in soda before you o...,1,8,2,1,0,0,0,0,1,False,numerical_only,,,,4
214,How much does it cost to set up a (living) trust?,My mom’s health is up in the air as she is in ...,0,r/legaladvice,how much does it cost to set up a (living) tru...,1,2,0,0,0,0,0,0,1,False,numerical_only,,,,4
215,Can I sue my landlord for not being able to ha...,I moved into this small until back in June it’...,0,r/legaladvice,can i sue my landlord for not being able to ha...,1,1,1,1,0,0,0,0,1,False,numerical_only,,,,4
235,CMV: Emergency sirens and car horns should not...,"First, some definitions\nBy “emergency sirens,...",1,r/changemyview,cmv: emergency sirens and car horns should not...,0,0,1,1,0,1,0,1,0,False,numerical_only,,,,4


In [75]:
errors_v1['openness'].value_counts()

openness
0    59
1    15
Name: count, dtype: int64

## Let's take samples of errors and start manual investigation on it and create error buckets

In [76]:
sample = errors_v1.sample(25, random_state=42)


In [77]:
sample

,title,body,openness,subreddit,text,has_question_mark,num_questions,uncertainty_count,has_uncertainty_marker,has_invitation_phrase,assertion_marker_count,has_reflective_phrase,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold
18,Do all flowering plants share a common ancesto...,With the variety of flowering plants in the wo...,0,r/askscience,do all flowering plants share a common ancesto...,1,5,0,0,0,0,0,0,1,False,numerical_only,,,,0
203,"Where do you organize ""run dates""?","I don't mean dates in a dating way, but i some...",0,r/running,"where do you organize ""run dates""? i don't mea...",1,3,0,0,0,0,0,0,1,False,numerical_only,,,,4
66,Fueling as a noob,Hi all I’m in need of some advice as a new run...,0,r/running,fueling as a noob hi all i’m in need of some a...,1,1,0,0,0,0,0,0,1,False,numerical_only,,,,1
12,Java or Javascript or C#?,"I'm not sure which one if these is better, to ...",0,r/learningprogramming,java or javascript or c#? i'm not sure which o...,1,2,1,1,0,0,0,0,1,False,numerical_only,,,,0
103,"If I don't get to sleep, so won't you (a hotel...",That was last 90s. We were four 18yos and we b...,0,r/prettyrevenge,"if i don't get to sleep, so won't you (a hotel...",1,5,0,0,0,0,0,0,1,False,numerical_only,,,,2
237,CMV: There is no actually good reason for ince...,I find incest disgusting like most people do. ...,1,r/changemyview,cmv: there is no actually good reason for ince...,0,0,0,0,0,0,0,1,0,False,numerical_only,,,,4
32,Would you rather have $1M right now or $1B if ...,The genie makes the offer. If you accept the $...,1,r/WouldYouRather,would you rather have $1m right now or $1b if ...,0,0,0,0,0,0,0,1,0,False,numerical_only,,,,0
113,Can I travel to US on (B1/B2) with expired Ger...,"Hello all,\nI would like to know if I can trav...",0,r/immigration,can i travel to us on (b1/b2) with expired ger...,1,2,0,0,0,0,0,0,1,False,numerical_only,,,,2
39,Does money bring happiness?,I was thinking about the question: Does money ...,1,r/TrueAskReddit,does money bring happiness? i was thinking abo...,1,4,0,0,1,1,0,1,0,False,numerical_only,,,,0
182,Do antinatalists have a positive duty to abort...,By abort I don't mean aborting a pregnancy spe...,1,r/askphilosophy,do antinatalists have a positive duty to abort...,1,2,1,1,0,0,0,1,0,False,numerical_only,,,,3


# Observation

- During error analysis, **74 misclassifications** were observed.
  - Only **15** belonged to the *positive (open)* class, despite a **balanced dataset**.
- Manual inspection of a **25-post error sample** revealed that:
  - ~**80%** of errors were driven by posts containing **multiple question marks**.

**Interpretation:**
- The model was **over-weighting the quantity of questions** as a proxy for openness.
- This led to **systematic false positives**, especially for:
  - Short questions  
  - Curiosity-only posts

---

# Action

- To correct this **surface-structure bias**:
  - The `num_questions` feature was **removed**
  - It was replaced with a **binary** `has_question_mark` indicator

**Goal:**
- Preserve *intent*  
- Eliminate *magnitude dominance*

---

# Rationale

- For **baseline openness detection**:
  - The *presence* of a question is a **sufficient signal**
- **Question intensity** is better handled by:
  - Semantic models  
  - Later pipeline stages


In [78]:
df.drop(columns=['num_questions'],axis=1,inplace=True)

In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   title                   240 non-null    object
 1   body                    240 non-null    object
 2   openness                240 non-null    int64 
 3   subreddit               240 non-null    object
 4   text                    240 non-null    object
 5   has_question_mark       240 non-null    int64 
 6   uncertainty_count       240 non-null    int64 
 7   has_uncertainty_marker  240 non-null    int64 
 8   has_invitation_phrase   240 non-null    int64 
 9   assertion_marker_count  240 non-null    int64 
 10  has_reflective_phrase   240 non-null    int64 
dtypes: int64(7), object(4)
memory usage: 20.8+ KB


# Re-running the experimenting and finding the error bucket

In [84]:
result_open_v2 = run_k_fold_error_analysis(df=df,label_col="openness")
print("num_questions" in df.columns)

False


In [86]:
errors_v2 = result_open_v2['errors_only']
errors_v2

,title,body,openness,subreddit,text,has_question_mark,uncertainty_count,has_uncertainty_marker,has_invitation_phrase,assertion_marker_count,has_reflective_phrase,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold
12,Java or Javascript or C#?,"I'm not sure which one if these is better, to ...",0,r/learningprogramming,java or javascript or c#? i'm not sure which o...,1,1,1,0,0,0,0,1,False,numerical_only,,,,0
14,Why are under cabinet strip lights so complica...,It seems crazy there is no consensus on how to...,0,r/HomeImprovement,why are under cabinet strip lights so complica...,1,0,0,0,0,0,0,1,False,numerical_only,,,,0
15,Is insulation between floors worth it during c...,my builder is charging $2500 for R-15 3.5 inch...,0,r/HomeImprovement,is insulation between floors worth it during c...,1,0,0,0,0,0,0,1,False,numerical_only,,,,0
17,Which companies now only offer on site softwar...,Hi all- currently work remotely and I now real...,0,r/cscareerquestions,which companies now only offer on site softwar...,1,0,0,0,0,0,0,1,False,numerical_only,,,,0
18,Do all flowering plants share a common ancesto...,With the variety of flowering plants in the wo...,0,r/askscience,do all flowering plants share a common ancesto...,1,0,0,0,0,0,0,1,False,numerical_only,,,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,How exactly is gas stored in soda before you o...,Hey scientists!\nMaybe this is a super basic q...,0,r/askscience,how exactly is gas stored in soda before you o...,1,2,1,0,0,0,0,1,False,numerical_only,,,,4
214,How much does it cost to set up a (living) trust?,My mom’s health is up in the air as she is in ...,0,r/legaladvice,how much does it cost to set up a (living) tru...,1,0,0,0,0,0,0,1,False,numerical_only,,,,4
215,Can I sue my landlord for not being able to ha...,I moved into this small until back in June it’...,0,r/legaladvice,can i sue my landlord for not being able to ha...,1,1,1,0,0,0,0,1,False,numerical_only,,,,4
235,CMV: Emergency sirens and car horns should not...,"First, some definitions\nBy “emergency sirens,...",1,r/changemyview,cmv: emergency sirens and car horns should not...,0,1,1,0,1,0,1,0,False,numerical_only,,,,4


In [87]:
errors_v2['openness'].value_counts()

openness
0    60
1    18
Name: count, dtype: int64

In [88]:
sample_v2 = errors_v2.sample(25, random_state=42)
sample_v2


,title,body,openness,subreddit,text,has_question_mark,uncertainty_count,has_uncertainty_marker,has_invitation_phrase,assertion_marker_count,has_reflective_phrase,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold
112,Bathroom Remodel Price,We are looking to have two bathrooms completel...,0,r/HomeImprovement,bathroom remodel price we are looking to have ...,1,1,1,0,0,0,0,1,False,numerical_only,,,,2
12,Java or Javascript or C#?,"I'm not sure which one if these is better, to ...",0,r/learningprogramming,java or javascript or c#? i'm not sure which o...,1,1,1,0,0,0,0,1,False,numerical_only,,,,0
113,Can I travel to US on (B1/B2) with expired Ger...,"Hello all,\nI would like to know if I can trav...",0,r/immigration,can i travel to us on (b1/b2) with expired ger...,1,0,0,0,0,0,0,1,False,numerical_only,,,,2
39,Does money bring happiness?,I was thinking about the question: Does money ...,1,r/TrueAskReddit,does money bring happiness? i was thinking abo...,1,0,0,1,1,0,1,0,False,numerical_only,,,,0
32,Would you rather have $1M right now or $1B if ...,The genie makes the offer. If you accept the $...,1,r/WouldYouRather,would you rather have $1m right now or $1b if ...,0,0,0,0,0,0,1,0,False,numerical_only,,,,0
237,CMV: There is no actually good reason for ince...,I find incest disgusting like most people do. ...,1,r/changemyview,cmv: there is no actually good reason for ince...,0,0,0,0,0,0,1,0,False,numerical_only,,,,4
107,conversion of despair,It’s 2016 again. Every emotion is a companion....,0,r/introvert,conversion of despair it’s 2016 again. every e...,1,1,1,0,0,0,0,1,False,numerical_only,,,,2
18,Do all flowering plants share a common ancesto...,With the variety of flowering plants in the wo...,0,r/askscience,do all flowering plants share a common ancesto...,1,0,0,0,0,0,0,1,False,numerical_only,,,,0
163,What is the exact definition of (or conditions...,Hi I was wondering what qualifies a system of ...,0,r/AskHistorians,what is the exact definition of (or conditions...,1,0,0,0,0,0,0,1,False,numerical_only,,,,3
203,"Where do you organize ""run dates""?","I don't mean dates in a dating way, but i some...",0,r/running,"where do you organize ""run dates""? i don't mea...",1,0,0,0,0,0,0,1,False,numerical_only,,,,4


## Observation:
The baseline openness model shows a strong false-positive bias, with ~77% of errors occurring when the true label is 0. Manual error analysis indicates that the model over-relies on surface-level cues such as question marks and uncertainty phrases. Posts that are factual, convergent, or expressive/rant-like—despite containing questions—are frequently misclassified as open. This suggests the model is capturing curiosity or uncertainty rather than genuine openness. Removing num_questions reduced reliance on punctuation but did not fully address the semantic mismatch, motivating feature redesign rather than further tuning of punctuation-based signals.

## Conclusion of this experiment:

The baseline openness model captures surface-level indicators such as explicit questions and uncertainty expressions. However, error analysis reveals a strong false-positive bias, particularly for factual, domain-convergent, and rhetorical questions embedded within narratives. This indicates that openness is not solely a function of interrogative or uncertain phrasing, but depends on semantic intent and discourse context. Consequently, further improvements require semantic representations beyond numerical structural features.

In [2]:
## Reference for feature_engineering_V1 for other experiments

import re

def openness_feature_engineer_v1(df):
    df = df.copy()

    # Drop unwanted columns (ignore if missing)
    df.drop(columns=['tag', 'effort', 'id', 'is_confident'], inplace=True, errors='ignore')

    # Combine text fields
    df["text_for_feature"] = df["title"].fillna("") + " " + df["body"].fillna("") 
    df["text"] = (df["title"].fillna("") + " " + df["body"].fillna("")).str.lower()

    # --- Question features ---
    def has_question_mark(text):
        if not isinstance(text, str):
            return 0
        return int("?" in text)

    def num_questions(text):
        if not isinstance(text, str):
            return 0
        return text.count("?")

    df["has_question_mark"] = df["text"].apply(has_question_mark)
    df["num_questions"] = df["text"].apply(num_questions)

    # --- Uncertainty features ---
    UNCERTAINTY_MARKERS = [
        "maybe", "might", "not sure", "i think", "i feel",
        "i guess", "i'm unsure", "i am unsure",
        "could be wrong", "not certain", "possibly"
    ]

    def uncertainty_count(text):
        if not isinstance(text, str):
            return 0
        return sum(1 for m in UNCERTAINTY_MARKERS if m in text)

    def has_uncertainty_marker(text):
        return int(uncertainty_count(text) > 0)

    df["uncertainty_count"] = df["text"].apply(uncertainty_count)
    df["has_uncertainty_marker"] = df["text"].apply(has_uncertainty_marker)

    # --- Invitation features ---
    INVITATION_PHRASES = [
        "what do you think",
        "any advice",
        "any thoughts",
        "can someone explain",
        "help me understand",
        "thoughts?",
        "does anyone know",
        "am i missing something"
    ]

    def has_invitation_phrase(text):
        if not isinstance(text, str):
            return 0
        return int(any(p in text for p in INVITATION_PHRASES))

    df["has_invitation_phrase"] = df["text"].apply(has_invitation_phrase)

    # --- Assertion features ---
    ASSERTION_MARKERS = [
        "obviously",
        "everyone knows",
        "the truth is",
        "there is no doubt",
        "clearly",
        "this proves",
        "it's obvious",
        "without question"
    ]

    def assertion_marker_count(text):
        if not isinstance(text, str):
            return 0
        return sum(1 for m in ASSERTION_MARKERS if m in text)

    df["assertion_marker_count"] = df["text"].apply(assertion_marker_count)

    # --- Reflective features ---
    REFLECTIVE_PHRASES = [
        "i'm trying to understand",
        "i am trying to understand",
        "i want to understand",
        "i'm learning",
        "i am learning",
        "i'm confused",
        "i am confused",
        "i don't fully understand",
        "i do not fully understand"
    ]

    def has_reflective_phrase(text):
        if not isinstance(text, str):
            return 0
        return int(any(p in text for p in REFLECTIVE_PHRASES))

    df["has_reflective_phrase"] = df["text"].apply(has_reflective_phrase)

    # Drop intermediate feature
    df.drop(columns=['num_questions'], inplace=True)

    return df
